In [1]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_DIR = "/content/drive/MyDrive/Pneumonia-Detection"

print(os.path.exists(BASE_DIR))
print(os.path.exists(f"{BASE_DIR}/data/train"))
print(os.path.exists(f"{BASE_DIR}/data/val"))
print(os.path.exists(f"{BASE_DIR}/data/test"))

True
True
True
True


In [4]:
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

BASE_DIR = "/content/drive/MyDrive/Pneumonia-Detection"

TRAIN_DIR = f"{BASE_DIR}/data/train"
VAL_DIR = f"{BASE_DIR}/data/val"
TEST_DIR = f"{BASE_DIR}/data/test"

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Found 624 files belonging to 2 classes.


In [5]:
from tensorflow.keras.applications.resnet50 import preprocess_input

In [6]:
def preprocess(images, labels):
    images = preprocess_input(images)
    return images, labels

train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)

In [7]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model_resnet50 = models.Sequential([
    layers.Input(shape=(224,224,3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model_resnet50.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,113 (90.98 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [8]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [9]:
model_resnet50.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [10]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [11]:
history_resnet50 = model_resnet50.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[early_stopping]
)

Epoch 1/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 1716s 10s/step - accuracy: 0.9325 - loss: 0.1824 - val_accuracy: 0.5625 - val_loss: 0.6922
Epoch 2/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 53s 324ms/step - accuracy: 0.9641 - loss: 0.0946 - val_accuracy: 0.8125 - val_loss: 0.3877
Epoch 3/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 54s 332ms/step - accuracy: 0.9682 - loss: 0.0826 - val_accuracy: 0.9375 - val_loss: 0.1299
Epoch 4/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 81s 328ms/step - accuracy: 0.9716 - loss: 0.0754 - val_accuracy: 0.8125 - val_loss: 0.2972
Epoch 5/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 53s 326ms/step - accuracy: 0.9753 - loss: 0.0646 - val_accuracy: 0.6875 - val_loss: 0.5424
Epoch 6/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 82s 329ms/step - accuracy: 0.9743 - loss: 0.0696 - val_accuracy: 0.8750 - val_loss: 0.2800
Epoch 7/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 52s 317ms/step - accuracy: 0.9781 - loss: 0.0603 - val_accuracy: 0.8750 - val_loss: 0.2221
Epoch 8/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 54s 328ms/step - accuracy: 0.9827 - loss: 0

In [12]:
test_loss_resnet50, test_accuracy_resnet50 = model_resnet50.evaluate(
    test_dataset
)

print(f"Test Loss: {test_loss_resnet50:.4f}")
print(f"Test Accuracy: {test_accuracy_resnet50:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 201s 10s/step - accuracy: 0.8077 - loss: 0.7885
Test Loss: 0.7885
Test Accuracy: 0.8077


### Fine-Tuning

In [13]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [14]:
print("Trainable Layers:")

for layer in base_model.layers[-10:]:
    print(layer.name, layer.trainable)

Trainable Layers:
conv5_block3_1_conv True
conv5_block3_1_bn True
conv5_block3_1_relu True
conv5_block3_2_conv True
conv5_block3_2_bn True
conv5_block3_2_relu True
conv5_block3_3_conv True
conv5_block3_3_bn True
conv5_block3_add True
conv5_block3_out True


In [15]:
from tensorflow.keras.optimizers import Adam

model_resnet50.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [16]:
history_resnet50_ft = model_resnet50.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[early_stopping]
) #Fine-tuning usually requires fewer epochs.

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 84s 366ms/step - accuracy: 0.9709 - loss: 0.0787 - val_accuracy: 1.0000 - val_loss: 0.1318
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 58s 354ms/step - accuracy: 0.9927 - loss: 0.0250 - val_accuracy: 1.0000 - val_loss: 0.0724
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 58s 352ms/step - accuracy: 0.9967 - loss: 0.0124 - val_accuracy: 0.9375 - val_loss: 0.1006
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 57s 346ms/step - accuracy: 0.9975 - loss: 0.0099 - val_accuracy: 1.0000 - val_loss: 0.0400
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 56s 345ms/step - accuracy: 0.9990 - loss: 0.0047 - val_accuracy: 1.0000 - val_loss: 0.0368
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 57s 344ms/step - accuracy: 0.9994 - loss: 0.0029 - val_accuracy: 0.9375 - val_loss: 0.1711
Epoch 7/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 82s 342ms/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 0.9375 - val_loss: 0.0798
Epoch 8/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 83s 347ms/step - accuracy: 0.9998 - loss: 0

In [17]:
test_loss_ft, test_accuracy_ft = model_resnet50.evaluate(
    test_dataset
)

print(f"Fine-Tuned Test Loss: {test_loss_ft:.4f}")
print(f"Fine-Tuned Test Accuracy: {test_accuracy_ft:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 294ms/step - accuracy: 0.8253 - loss: 1.0488
Fine-Tuned Test Loss: 1.0488
Fine-Tuned Test Accuracy: 0.8253


In [19]:
import os

MODEL_DIR = "/content/drive/MyDrive/Pneumonia-Detection/models"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Exists:", os.path.exists(MODEL_DIR))

Exists: True


In [20]:
model_resnet50.save(
    "/content/drive/MyDrive/Pneumonia-Detection/models/resnet50_finetuned.keras"
)

print("Model Saved Successfully")

Model Saved Successfully
